# GEAR Metal-Only Ultimate v2

**Metal-binding residue prediction only — feature discovery first, architecture second, external confirmation last.**

This notebook is intentionally narrower than the original `GEAR_ULTIMATE_METAL_BINDING.ipynb`.

## Scientific contract

**Development set:** DevSet1010, official bindNode24 protein-level 5-fold split  
**External confirmation:** TestSet300 + TestSetNew46  
**Primary metric:** residue-level AUPRC  
**Secondary metrics:** MCC, F1, AUROC  
**Base learner for feature discovery:** MLP only  
**Selection source:** DevSet1010 only  
**External sets:** opened only after feature/model choices are locked

The workflow is:

```text
ESM-2 base
   │
   ├── Stage A: family screen with the SAME MLP
   │       Structure / ATOM / CHEM / MECH1 / MULTI2 / SEQCTX
   │
   ├── Stage B: component screen conditioned on ESM+ATOM+CHEM
   │       ACCESS / SHELL2 / ENV / CONF
   │       METALSPEC / ELEC3D / TOPO / SOLV / SEQMOTIF
   │
   ├── Stage C: combination ladder with the SAME MLP
   │
   ├── LOCK representation from Dev CV only
   │
   ├── Stage D: architecture/model-family screen
   │       MLP / KAN / XGBoost
   │
   ├── optional graph diagnostic
   │
   └── FINAL: train five official folds and evaluate once on
           TestSet300 + TestSetNew46
```

This design separates **representation gain** from **architecture gain**.  
A feature is not credited simply because a more powerful classifier was used at the same time.

All results are persisted under a dedicated run family in Google Drive and compatible feature caches from the existing GEAR project are reused.


## 0. What the previous notebook already established

The uploaded historical notebook supports the following observations:

| Representation | Dev CV AUPRC |
|---|---:|
| ESM | ~0.4023 |
| ESM + Structure | ~0.4049 |
| ESM + ATOM | ~0.4393 |
| ESM + CHEM | ~0.4431 |
| ESM + ATOM + CHEM | ~0.4473 |
| ESM + ATOM + CHEM + MULTI2 | ~0.4584 |

Historical external AUPRC:

| Representation | Test300 | Test46 |
|---|---:|---:|
| Historical ESM+Structure+KAN | 0.3110 | 0.5634 |
| ESM+ATOM | 0.3578 | 0.6520 |
| ESM+CHEM | 0.3525 | 0.6784 |
| ESM+ATOM+CHEM | 0.3694 | 0.6655 |
| ESM+MULTI2 | 0.3718 | 0.6641 |
| **ESM+ATOM+CHEM+MULTI2** | **0.3842** | **0.6936** |

The new notebook does **not** simply assume that the historical winner is final.  
It reruns the representation question cleanly on **DevSet1010 with one fixed MLP**, records the decision, and only then opens the external sets.


In [ ]:
# ============================================================
# 1. DRIVE / PATHS / DEPENDENCIES
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, inspect, platform, subprocess, math
import numpy as np
import pandas as pd
import torch

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
ULTIMATE_DIR = BASE/'GEAR_ULTIMATE'

if not ULTIMATE_DIR.exists():
    raise FileNotFoundError(
        f'{ULTIMATE_DIR} not found. Keep the existing GEAR_ULTIMATE backend on Drive.'
    )

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(ULTIMATE_DIR/'requirements.txt')
])

sys.path.insert(0, str(ULTIMATE_DIR))

from gear_ultimate import __version__ as GEAR_ULTIMATE_VERSION
from gear_ultimate import gear_core as core
from gear_ultimate import gear_structure as structure
from gear_ultimate import gear_mechanistic as mech
from gear_ultimate import gear_graph as graph
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RUN_ROOT = BASE/'results'/'GEAR_METAL_ONLY_ULTIMATE_V2'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('BASE:', BASE)
print('ULTIMATE_DIR:', ULTIMATE_DIR)
print('RUN_ROOT:', RUN_ROOT)
print('GEAR version:', GEAR_ULTIMATE_VERSION)
print('Torch:', torch.__version__)
print('Device:', DEVICE)
print('KAN available:', core.HAS_KAN)


Mounted at /content/drive
BASE: /content/drive/MyDrive/Protein-binding-Phi
ULTIMATE_DIR: /content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE
RUN_ROOT: /content/drive/MyDrive/Protein-binding-Phi/results/GEAR_METAL_ONLY_ULTIMATE_V2
GEAR version: 1.0.0
Torch: 2.11.0+cu128
Device: cuda
KAN available: True


In [ ]:
from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive/Protein-binding-Phi")

print("=== GPU ===")
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("\n=== EXISTING FEATURE CACHE ===")

CACHE = BASE / "cache"

if CACHE.exists():
    files = [p for p in CACHE.rglob("*") if p.is_file()]
    print("Cache files:", len(files))
    print(
        "Total cache:",
        round(sum(p.stat().st_size for p in files)/1024**3, 2),
        "GB"
    )

    keywords = [
        "esm",
        "spatial",
        "local",
        "atom",
        "chem",
        "access",
        "shell",
        "multi",
        "seqctx",
    ]

    for key in keywords:
        hits = [
            p for p in files
            if key.lower() in str(p).lower()
        ]
        size = sum(p.stat().st_size for p in hits)
        print(
            f"{key:10s}: "
            f"{len(hits):5d} files | "
            f"{size/1024**3:.3f} GB"
        )

print("\n=== NEW V2 EXPERIMENT CACHE ===")

versions = [
    "metal_v2_01_family_mlp",
    "metal_v2_02_component_mlp",
    "metal_v2_03_combo_mlp",
    "metal_v2_04_locked_model_family",
    "metal_v2_05_graph_diagnostic",
    "metal_v2_06_final_locked_external",
]

for v in versions:
    d = BASE / "results" / v / "metal_binding"

    if not d.exists():
        print(f"{v:42s}: NOT STARTED")
        continue

    fs = [p for p in d.rglob("*") if p.is_file()]

    print(
        f"{v:42s}: "
        f"{len(fs)} files | "
        f"{sum(p.stat().st_size for p in fs)/1024**2:.1f} MB"
    )

=== GPU ===
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB

=== EXISTING FEATURE CACHE ===
Cache files: 12437
Total cache: 8.05 GB
esm       :  1359 files | 0.747 GB
spatial   :    21 files | 0.825 GB
local     :     4 files | 0.126 GB
atom      :     3 files | 0.013 GB
chem      :     3 files | 0.021 GB
access    :     0 files | 0.000 GB
shell     :     0 files | 0.000 GB
multi     :     3 files | 0.052 GB
seqctx    :     0 files | 0.000 GB

=== NEW V2 EXPERIMENT CACHE ===
metal_v2_01_family_mlp                    : NOT STARTED
metal_v2_02_component_mlp                 : NOT STARTED
metal_v2_03_combo_mlp                     : NOT STARTED
metal_v2_04_locked_model_family           : NOT STARTED
metal_v2_05_graph_diagnostic              : NOT STARTED
metal_v2_06_final_locked_external         : NOT STARTED


## 2. Master switches

Default settings run the complete **tabular scientific pipeline**:

1. feature-family MLP screen;
2. mechanistic component MLP screen;
3. combination ladder;
4. Dev-only representation lock;
5. same-representation MLP/KAN/XGBoost comparison;
6. final external confirmation.

The graph branch is deliberately **off by default** because it is a second research question and can be considerably more expensive. Turn it on after inspecting the locked tabular representation.

`FORCE_RERUN=False` is the normal setting. Existing backend caches/run outputs remain reusable.


In [ ]:
# ============================================================
# 2. MASTER SWITCHES
# ============================================================

PREPARE_CONTEXT = True
ESM_BATCH_SIZE = 8

RUN_FAMILY_SCREEN = True
RUN_COMPONENT_SCREEN = True
RUN_COMBINATION_LADDER = True
RUN_MODEL_FAMILY_SCREEN = True

# Graph is a second-stage diagnostic. Keep False for the first run.
RUN_GRAPH_DIAGNOSTIC = False
RUN_ESM_GRAPH = False

# Once representation/model are selected from Dev, this evaluates Test300/Test46.
RUN_FINAL_EXTERNAL = True

# Never use Test300/Test46 to choose features below.
SELECTION_POLICY = 'best_mean'       # 'best_mean' or 'one_se'
FORCE_RERUN = False

# Version names are fixed so interrupted runs can resume.
RUN_VERSION_FAMILY = 'metal_v2_01_family_mlp'
RUN_VERSION_COMPONENT = 'metal_v2_02_component_mlp'
RUN_VERSION_COMBO = 'metal_v2_03_combo_mlp'
RUN_VERSION_MODEL = 'metal_v2_04_locked_model_family'
RUN_VERSION_GRAPH = 'metal_v2_05_graph_diagnostic'
RUN_VERSION_FINAL = 'metal_v2_06_final_locked_external'

print('Selection policy:', SELECTION_POLICY)
print('Graph diagnostic:', RUN_GRAPH_DIAGNOSTIC)
print('Final external:', RUN_FINAL_EXTERNAL)


Selection policy: best_mean
Graph diagnostic: False
Final external: True


## 3. Build/reuse the unified context

Feature blocks in the recovered GEAR backend:

- `AA20`
- `ESM1280`: ESM-2 `esm2_t33_650M_UR50D`, mean of the last two hidden layers
- `SPATIAL112`
- `LOCAL8`
- `ATOM15`
- `CHEM24`
- `ACCESS10`
- `SHELL2_12`
- `ENV8`
- `CONF4`
- `METALSPEC18`
- `ELEC3D8`
- `TOPO12`
- `SOLV10`
- `SEQMOTIF10`
- `SEQCTX182`

Grouped mechanisms:

```text
STRUCTURE = SPATIAL + LOCAL          = 120 d

MECH1     = ACCESS + SHELL2
            + ENV + CONF             = 34 d

MULTI2    = METALSPEC + ELEC3D
            + TOPO + SOLV
            + SEQMOTIF               = 58 d
```

`ATOM` and `CHEM` are kept separate because the historical runs already suggest that both carry strong but non-identical signal.


In [ ]:
# ============================================================
# 3. PREPARE / REUSE CONTEXT
# ============================================================
ctx = suite.prepare_context(
    base=BASE,
    run_version='metal_v2_context',
    models=('MLP','KAN','XGBoost'),
    ensure_base=True,
    add_mech=True,
    add_multi=True,
    add_seqctx=True,
    esm_batch_size=ESM_BATCH_SIZE,
    device=DEVICE,
)

display(suite.audit_context(ctx))
display(reporting.dimension_table(ctx.stores['train']))

# Label-independent leakage audit.
explicit_blocks = [
    'AA','SPATIAL','LOCAL','ATOM','CHEM','ACCESS','SHELL2','ENV','CONF',
    'METALSPEC','ELEC3D','TOPO','SOLV','SEQMOTIF','SEQCTX'
]
display(reporting.validate_no_direct_label_features(ctx.stores, explicit_blocks))


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

,split,proteins,residues,positives,prevalence
0,train,1010,170181,2370,0.013926
1,test300,300,62689,881,0.014054
2,test46,46,6227,77,0.012366


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
3,LOCAL,8
10,METALSPEC,18


,split,block,missing_proteins,mean_finite_fraction
0,train,AA,0,1.0
1,train,SPATIAL,0,1.0
2,train,LOCAL,0,1.0
3,train,ATOM,0,1.0
4,train,CHEM,0,1.0
5,train,ACCESS,0,1.0
6,train,SHELL2,0,1.0
7,train,ENV,0,1.0
8,train,CONF,0,1.0
9,train,METALSPEC,0,1.0


In [ ]:
# ============================================================
# 4. FEATURE CONTRACT
# ============================================================
BLOCK_DIMS = {
    'AA':20,
    'ESM':1280,
    'SPATIAL':112,
    'LOCAL':8,
    'ATOM':15,
    'CHEM':24,
    'ACCESS':10,
    'SHELL2':12,
    'ENV':8,
    'CONF':4,
    'METALSPEC':18,
    'ELEC3D':8,
    'TOPO':12,
    'SOLV':10,
    'SEQMOTIF':10,
    'SEQCTX':182,
}

FAMILIES = {
    'Structure': ['SPATIAL','LOCAL'],
    'Atomic': ['ATOM'],
    'Chem': ['CHEM'],
    'MECH1': ['ACCESS','SHELL2','ENV','CONF'],
    'MULTI2': ['METALSPEC','ELEC3D','TOPO','SOLV','SEQMOTIF'],
    'SeqCtx': ['SEQCTX'],
}

MECHANISTIC_COMPONENTS = {
    'ACCESS': ['ACCESS'],
    'SHELL2': ['SHELL2'],
    'ENV': ['ENV'],
    'CONF': ['CONF'],
    'METALSPEC': ['METALSPEC'],
    'ELEC3D': ['ELEC3D'],
    'TOPO': ['TOPO'],
    'SOLV': ['SOLV'],
    'SEQMOTIF': ['SEQMOTIF'],
}

def block_dim(blocks):
    return int(sum(BLOCK_DIMS[b] for b in blocks))

feature_inventory = []
for fam, blocks in FAMILIES.items():
    feature_inventory.append({
        'family': fam,
        'blocks': '+'.join(blocks),
        'dim': block_dim(blocks),
    })

display(pd.DataFrame(feature_inventory))

print('Backend MULTI2:', getattr(suite, 'MULTI2', 'not exposed'))
print('Official fold count:', len(ctx.fold_ids))


,family,blocks,dim
0,Structure,SPATIAL+LOCAL,120
1,Atomic,ATOM,15
2,Chem,CHEM,24
3,MECH1,ACCESS+SHELL2+ENV+CONF,34
4,MULTI2,METALSPEC+ELEC3D+TOPO+SOLV+SEQMOTIF,58
5,SeqCtx,SEQCTX,182


Backend MULTI2: ['METALSPEC', 'ELEC3D', 'TOPO', 'SOLV', 'SEQMOTIF']
Official fold count: 5


# Stage A — MLP family screen

This is the cleanest answer to:

> **Starting from exactly the same ESM representation and exactly the same MLP, which biological/structural family adds signal?**

Every candidate differs from `ESM` only by the added feature family.

No KAN, XGBoost, graph, or external set is allowed to influence this stage.


In [ ]:
# ============================================================
# 5. STAGE A — DEV-ONLY FAMILY SCREEN, MLP FIXED
# ============================================================
FAMILY_GRID = {
    'ESM': ['ESM'],
    'ESM+Structure': ['ESM', *FAMILIES['Structure']],
    'ESM+Atomic': ['ESM', *FAMILIES['Atomic']],
    'ESM+Chem': ['ESM', *FAMILIES['Chem']],
    'ESM+MECH1': ['ESM', *FAMILIES['MECH1']],
    'ESM+MULTI2': ['ESM', *FAMILIES['MULTI2']],
    'ESM+SeqCtx': ['ESM', *FAMILIES['SeqCtx']],
}

if RUN_FAMILY_SCREEN:
    family_cv, family_folds = suite.run_dev_cv_only(
        ctx,
        FAMILY_GRID,
        models=('MLP',),
        run_version=RUN_VERSION_FAMILY,
    )
    display(family_cv.sort_values('CV_AUPRC', ascending=False))
    family_cv.to_csv(RUN_ROOT/'family_screen_cv.csv', index=False)
    family_folds.to_csv(RUN_ROOT/'family_screen_folds.csv', index=False)
else:
    family_cv = pd.read_csv(RUN_ROOT/'family_screen_cv.csv')
    family_folds = pd.read_csv(RUN_ROOT/'family_screen_folds.csv')

family_rank = family_cv.copy()
base_score = float(
    family_rank.loc[family_rank['representation'].eq('ESM'), 'CV_AUPRC'].iloc[0]
)
family_rank['delta_vs_ESM'] = family_rank['CV_AUPRC'] - base_score
family_rank['feature_dim'] = family_rank['representation'].map(
    {k:block_dim(v) for k,v in FAMILY_GRID.items()}
)
display(
    family_rank.sort_values(
        ['delta_vs_ESM','CV_AUPRC'],
        ascending=False
    )
)
family_rank.to_csv(RUN_ROOT/'family_screen_ranked.csv', index=False)


DONE MLP ESM 1 AUPRC 0.40043785980878355
DONE MLP ESM 2 AUPRC 0.42928827158112737
DONE MLP ESM 3 AUPRC 0.448956744353674
DONE MLP ESM 4 AUPRC 0.4139775478691723
DONE MLP ESM 5 AUPRC 0.3188716616637845
DONE MLP ESM+Structure 1 AUPRC 0.3987528917614856
DONE MLP ESM+Structure 2 AUPRC 0.40465925593515617
DONE MLP ESM+Structure 3 AUPRC 0.4415713934344139
DONE MLP ESM+Structure 4 AUPRC 0.42437737988579904
DONE MLP ESM+Structure 5 AUPRC 0.35500745780120657
DONE MLP ESM+Atomic 1 AUPRC 0.4298756775434681
DONE MLP ESM+Atomic 2 AUPRC 0.4552918763725826
DONE MLP ESM+Atomic 3 AUPRC 0.4618501293061454
DONE MLP ESM+Atomic 4 AUPRC 0.4633816048740042
DONE MLP ESM+Atomic 5 AUPRC 0.3858811921178556
DONE MLP ESM+Chem 1 AUPRC 0.43662429970363215
DONE MLP ESM+Chem 2 AUPRC 0.44601088789843407
DONE MLP ESM+Chem 3 AUPRC 0.47673738635739565
DONE MLP ESM+Chem 4 AUPRC 0.4614206103725971
DONE MLP ESM+Chem 5 AUPRC 0.3949699403510923
DONE MLP ESM+MECH1 1 AUPRC 0.39043474705609
DONE MLP ESM+MECH1 2 AUPRC 0.4320286912

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
4,MLP,ESM+MULTI2,0.449412,0.024290,0.880684
2,MLP,ESM+Chem,0.443153,0.030960,0.875456
1,MLP,ESM+Atomic,0.439256,0.032736,0.870191
3,MLP,ESM+MECH1,0.405819,0.049078,0.874953
6,MLP,ESM+Structure,0.404874,0.032601,0.857461
0,MLP,ESM,0.402306,0.050013,0.867962
5,MLP,ESM+SeqCtx,0.394035,0.045240,0.862486


,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_ESM,feature_dim
4,MLP,ESM+MULTI2,0.449412,0.024290,0.880684,0.047106,1338
2,MLP,ESM+Chem,0.443153,0.030960,0.875456,0.040846,1304
1,MLP,ESM+Atomic,0.439256,0.032736,0.870191,0.036950,1295
3,MLP,ESM+MECH1,0.405819,0.049078,0.874953,0.003512,1314
6,MLP,ESM+Structure,0.404874,0.032601,0.857461,0.002567,1400
0,MLP,ESM,0.402306,0.050013,0.867962,0.000000,1280
5,MLP,ESM+SeqCtx,0.394035,0.045240,0.862486,-0.008272,1462


# Stage B — Mechanistic component screen

Family-level gain is useful, but it does not tell us **which part of the mechanism is carrying the signal**.

This screen conditions on the strongest historical compact core:

```text
ESM + ATOM + CHEM
```

and adds **one component at a time**.

This is much more informative than testing `TOPO`, `SOLV`, etc. on top of weak AA-only features: it asks whether each descriptor contains **marginal information beyond ESM + coordination geometry + chemistry**.


In [ ]:
# ============================================================
# 6. STAGE B — COMPONENT SCREEN CONDITIONAL ON ESM+ATOM+CHEM
# ============================================================
CORE_BLOCKS = ['ESM','ATOM','CHEM']

COMPONENT_GRID = {
    'Core_ESM+Atomic+Chem': CORE_BLOCKS,
}

for name, blocks in MECHANISTIC_COMPONENTS.items():
    COMPONENT_GRID[f'Core+{name}'] = [*CORE_BLOCKS, *blocks]

# Two grouped controls.
COMPONENT_GRID['Core+MECH1'] = [*CORE_BLOCKS, *FAMILIES['MECH1']]
COMPONENT_GRID['Core+MULTI2'] = [*CORE_BLOCKS, *FAMILIES['MULTI2']]

if RUN_COMPONENT_SCREEN:
    component_cv, component_folds = suite.run_dev_cv_only(
        ctx,
        COMPONENT_GRID,
        models=('MLP',),
        run_version=RUN_VERSION_COMPONENT,
    )
    display(component_cv.sort_values('CV_AUPRC', ascending=False))
    component_cv.to_csv(RUN_ROOT/'component_screen_cv.csv', index=False)
    component_folds.to_csv(RUN_ROOT/'component_screen_folds.csv', index=False)
else:
    component_cv = pd.read_csv(RUN_ROOT/'component_screen_cv.csv')
    component_folds = pd.read_csv(RUN_ROOT/'component_screen_folds.csv')

core_score = float(
    component_cv.loc[
        component_cv['representation'].eq('Core_ESM+Atomic+Chem'),
        'CV_AUPRC'
    ].iloc[0]
)

component_rank = component_cv.copy()
component_rank['delta_vs_core'] = component_rank['CV_AUPRC'] - core_score
component_rank['feature_dim'] = component_rank['representation'].map(
    {k:block_dim(v) for k,v in COMPONENT_GRID.items()}
)

display(component_rank.sort_values('delta_vs_core', ascending=False))
component_rank.to_csv(RUN_ROOT/'component_screen_ranked.csv', index=False)


DONE MLP Core_ESM+Atomic+Chem 1 AUPRC 0.443930648730958
DONE MLP Core_ESM+Atomic+Chem 2 AUPRC 0.4547174389916122
DONE MLP Core_ESM+Atomic+Chem 3 AUPRC 0.47609359293510856
DONE MLP Core_ESM+Atomic+Chem 4 AUPRC 0.4612982801883095
DONE MLP Core_ESM+Atomic+Chem 5 AUPRC 0.40033715114632595
DONE MLP Core+ACCESS 1 AUPRC 0.4498263522563528
DONE MLP Core+ACCESS 2 AUPRC 0.45793760597522815
DONE MLP Core+ACCESS 3 AUPRC 0.47762087927560015
DONE MLP Core+ACCESS 4 AUPRC 0.4722498726579979
DONE MLP Core+ACCESS 5 AUPRC 0.40258974570941863
DONE MLP Core+SHELL2 1 AUPRC 0.4533219694835629
DONE MLP Core+SHELL2 2 AUPRC 0.46820880332826803
DONE MLP Core+SHELL2 3 AUPRC 0.4680872131193137
DONE MLP Core+SHELL2 4 AUPRC 0.4728478619091882
DONE MLP Core+SHELL2 5 AUPRC 0.408266706448118
DONE MLP Core+ENV 1 AUPRC 0.4478367732030667
DONE MLP Core+ENV 2 AUPRC 0.44745676136994794
DONE MLP Core+ENV 3 AUPRC 0.4811774005029881
DONE MLP Core+ENV 4 AUPRC 0.4761095820273421
DONE MLP Core+ENV 5 AUPRC 0.3924816378031443
DONE 

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
6,MLP,Core+MULTI2,0.458725,0.023387,0.875499
10,MLP,Core+TOPO,0.458587,0.025569,0.868254
8,MLP,Core+SHELL2,0.454147,0.026681,0.872956
0,MLP,Core+ACCESS,0.452045,0.029782,0.873630
9,MLP,Core+SOLV,0.451795,0.028208,0.875942
4,MLP,Core+MECH1,0.451778,0.030730,0.881179
5,MLP,Core+METALSPEC,0.451370,0.031969,0.879128
7,MLP,Core+SEQMOTIF,0.450904,0.027749,0.875736
2,MLP,Core+ELEC3D,0.450044,0.034462,0.878103
3,MLP,Core+ENV,0.449012,0.035243,0.873918


,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_core,feature_dim
6,MLP,Core+MULTI2,0.458725,0.023387,0.875499,0.011450,1377
10,MLP,Core+TOPO,0.458587,0.025569,0.868254,0.011312,1331
8,MLP,Core+SHELL2,0.454147,0.026681,0.872956,0.006871,1331
0,MLP,Core+ACCESS,0.452045,0.029782,0.873630,0.004769,1329
9,MLP,Core+SOLV,0.451795,0.028208,0.875942,0.004519,1329
4,MLP,Core+MECH1,0.451778,0.030730,0.881179,0.004503,1353
5,MLP,Core+METALSPEC,0.451370,0.031969,0.879128,0.004094,1337
7,MLP,Core+SEQMOTIF,0.450904,0.027749,0.875736,0.003628,1329
2,MLP,Core+ELEC3D,0.450044,0.034462,0.878103,0.002769,1327
3,MLP,Core+ENV,0.449012,0.035243,0.873918,0.001737,1327


# Stage C — Combination ladder

Now we test **combinations**, still with the same MLP.

The ladder intentionally contains:

- known strong compact combinations;
- grouped mechanistic additions;
- coarse-structure and sequence-context controls;
- a maximal tabular combination as a redundancy check.

This stage is the only place from which the final **representation** is locked.


In [ ]:
# ============================================================
# 7. STAGE C — DEV-ONLY COMBINATION LADDER, MLP FIXED
# ============================================================
COMBO_GRID = {
    'ESM': ['ESM'],
    'ESM+Atomic': ['ESM','ATOM'],
    'ESM+Chem': ['ESM','CHEM'],
    'ESM+Atomic+Chem': ['ESM','ATOM','CHEM'],

    'ESM+Atomic+Chem+MECH1':
        ['ESM','ATOM','CHEM', *FAMILIES['MECH1']],

    'ESM+Atomic+Chem+MULTI2':
        ['ESM','ATOM','CHEM', *FAMILIES['MULTI2']],

    'ESM+Atomic+Chem+MECH1+MULTI2':
        ['ESM','ATOM','CHEM', *FAMILIES['MECH1'], *FAMILIES['MULTI2']],

    'ESM+Atomic+Chem+MULTI2+Structure':
        ['ESM','ATOM','CHEM', *FAMILIES['MULTI2'], *FAMILIES['Structure']],

    'ESM+Atomic+Chem+MULTI2+SeqCtx':
        ['ESM','ATOM','CHEM', *FAMILIES['MULTI2'], *FAMILIES['SeqCtx']],

    'ESM+Atomic+Chem+MECH1+MULTI2+Structure':
        ['ESM','ATOM','CHEM',
         *FAMILIES['MECH1'], *FAMILIES['MULTI2'], *FAMILIES['Structure']],

    'ESM+Atomic+Chem+MECH1+MULTI2+SeqCtx':
        ['ESM','ATOM','CHEM',
         *FAMILIES['MECH1'], *FAMILIES['MULTI2'], *FAMILIES['SeqCtx']],

    'ESM+ALL_EXPLICIT':
        ['ESM','ATOM','CHEM',
         *FAMILIES['MECH1'], *FAMILIES['MULTI2'],
         *FAMILIES['Structure'], *FAMILIES['SeqCtx']],
}

if RUN_COMBINATION_LADDER:
    combo_cv, combo_folds = suite.run_dev_cv_only(
        ctx,
        COMBO_GRID,
        models=('MLP',),
        run_version=RUN_VERSION_COMBO,
    )
    display(combo_cv.sort_values('CV_AUPRC', ascending=False))
    combo_cv.to_csv(RUN_ROOT/'combo_ladder_cv.csv', index=False)
    combo_folds.to_csv(RUN_ROOT/'combo_ladder_folds.csv', index=False)
else:
    combo_cv = pd.read_csv(RUN_ROOT/'combo_ladder_cv.csv')
    combo_folds = pd.read_csv(RUN_ROOT/'combo_ladder_folds.csv')


DONE MLP ESM 1 AUPRC 0.40043785980878355
DONE MLP ESM 2 AUPRC 0.42928827158112737
DONE MLP ESM 3 AUPRC 0.448956744353674
DONE MLP ESM 4 AUPRC 0.4139775478691723
DONE MLP ESM 5 AUPRC 0.3188716616637845
DONE MLP ESM+Atomic 1 AUPRC 0.4298756775434681
DONE MLP ESM+Atomic 2 AUPRC 0.4552918763725826
DONE MLP ESM+Atomic 3 AUPRC 0.4618501293061454
DONE MLP ESM+Atomic 4 AUPRC 0.4633816048740042
DONE MLP ESM+Atomic 5 AUPRC 0.3858811921178556
DONE MLP ESM+Chem 1 AUPRC 0.43662429970363215
DONE MLP ESM+Chem 2 AUPRC 0.44601088789843407
DONE MLP ESM+Chem 3 AUPRC 0.47673738635739565
DONE MLP ESM+Chem 4 AUPRC 0.4614206103725971
DONE MLP ESM+Chem 5 AUPRC 0.3949699403510923
DONE MLP ESM+Atomic+Chem 1 AUPRC 0.443930648730958
DONE MLP ESM+Atomic+Chem 2 AUPRC 0.4547174389916122
DONE MLP ESM+Atomic+Chem 3 AUPRC 0.47609359293510856
DONE MLP ESM+Atomic+Chem 4 AUPRC 0.4612982801883095
DONE MLP ESM+Atomic+Chem 5 AUPRC 0.40033715114632595
DONE MLP ESM+Atomic+Chem+MECH1 1 AUPRC 0.4554934851563227
DONE MLP ESM+Atom

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
8,MLP,ESM+Atomic+Chem+MULTI2,0.458725,0.023387,0.875499
6,MLP,ESM+Atomic+Chem+MECH1+MULTI2+SeqCtx,0.456484,0.029812,0.880569
5,MLP,ESM+Atomic+Chem+MECH1+MULTI2,0.456407,0.031176,0.883277
10,MLP,ESM+Atomic+Chem+MULTI2+Structure,0.455167,0.027536,0.874315
9,MLP,ESM+Atomic+Chem+MULTI2+SeqCtx,0.454201,0.031020,0.880942
7,MLP,ESM+Atomic+Chem+MECH1+MULTI2+Structure,0.452418,0.032149,0.882430
1,MLP,ESM+ALL_EXPLICIT,0.451876,0.030212,0.877515
4,MLP,ESM+Atomic+Chem+MECH1,0.451778,0.030730,0.881179
3,MLP,ESM+Atomic+Chem,0.447275,0.028709,0.872497
11,MLP,ESM+Chem,0.443153,0.030960,0.875456


## 8. Representation lock — Dev only

Two choices are reported:

### Best-mean rule
Select the representation with the highest Dev CV AUPRC.

### One-standard-error rule
If fold-level metrics are available, identify all representations within one SE of the best mean and select the **lowest-dimensional** one.

The notebook saves both. `SELECTION_POLICY` determines which one becomes the locked representation.

This makes the choice explicit and reproducible instead of silently choosing the model that happened to look best on Test300/Test46.


In [ ]:
# ============================================================
# 8. LOCK REPRESENTATION USING DEV ONLY
# ============================================================
def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def representation_stats_from_folds(fold_df):
    rep_col = _find_col(
        fold_df,
        ['representation','experiment','feature_set','rep']
    )
    score_col = _find_col(
        fold_df,
        ['AUPRC','val_AUPRC','CV_AUPRC','auprc']
    )
    fold_col = _find_col(
        fold_df,
        ['fold','fold_id','split']
    )

    if rep_col is None or score_col is None:
        return None

    q = (
        fold_df
        .groupby(rep_col)[score_col]
        .agg(['mean','std','count'])
        .reset_index()
        .rename(columns={rep_col:'representation'})
    )
    q['se'] = q['std'] / np.sqrt(q['count'].clip(lower=1))
    return q

# Best mean from the official CV summary.
combo_rank = combo_cv.copy()
combo_rank['feature_dim'] = combo_rank['representation'].map(
    {k:block_dim(v) for k,v in COMBO_GRID.items()}
)
combo_rank = combo_rank.sort_values('CV_AUPRC', ascending=False).reset_index(drop=True)

BEST_MEAN_REP = str(combo_rank.iloc[0]['representation'])

fold_stats = representation_stats_from_folds(combo_folds)

if fold_stats is not None and not fold_stats.empty:
    fold_stats['feature_dim'] = fold_stats['representation'].map(
        {k:block_dim(v) for k,v in COMBO_GRID.items()}
    )

    best_row = fold_stats.sort_values('mean', ascending=False).iloc[0]
    cutoff = float(best_row['mean'] - best_row['se'])

    eligible = fold_stats[
        fold_stats['mean'] >= cutoff
    ].copy()

    eligible = eligible.sort_values(
        ['feature_dim','mean'],
        ascending=[True,False]
    )

    ONE_SE_REP = str(eligible.iloc[0]['representation'])
else:
    ONE_SE_REP = BEST_MEAN_REP

if SELECTION_POLICY == 'best_mean':
    LOCKED_REP = BEST_MEAN_REP
elif SELECTION_POLICY == 'one_se':
    LOCKED_REP = ONE_SE_REP
else:
    raise ValueError("SELECTION_POLICY must be 'best_mean' or 'one_se'")

LOCKED_BLOCKS = list(COMBO_GRID[LOCKED_REP])

decision_table = combo_rank[
    ['representation','CV_AUPRC','feature_dim']
].copy()

display(decision_table)
if fold_stats is not None:
    print('\nFold-level representation statistics:')
    display(fold_stats.sort_values('mean', ascending=False))

print('\nBEST_MEAN_REP =', BEST_MEAN_REP)
print('ONE_SE_REP   =', ONE_SE_REP)
print('LOCK POLICY  =', SELECTION_POLICY)
print('LOCKED_REP   =', LOCKED_REP)
print('LOCKED_BLOCKS=', LOCKED_BLOCKS)
print('LOCKED_DIM   =', block_dim(LOCKED_BLOCKS))

representation_decision = {
    'selection_policy': SELECTION_POLICY,
    'best_mean_representation': BEST_MEAN_REP,
    'one_se_representation': ONE_SE_REP,
    'locked_representation': LOCKED_REP,
    'locked_blocks': LOCKED_BLOCKS,
    'locked_dimension': block_dim(LOCKED_BLOCKS),
}

with open(RUN_ROOT/'locked_representation.json', 'w') as f:
    json.dump(representation_decision, f, indent=2)


,representation,CV_AUPRC,feature_dim
0,ESM+Atomic+Chem+MULTI2,0.458725,1377
1,ESM+Atomic+Chem+MECH1+MULTI2+SeqCtx,0.456484,1593
2,ESM+Atomic+Chem+MECH1+MULTI2,0.456407,1411
3,ESM+Atomic+Chem+MULTI2+Structure,0.455167,1497
4,ESM+Atomic+Chem+MULTI2+SeqCtx,0.454201,1559
5,ESM+Atomic+Chem+MECH1+MULTI2+Structure,0.452418,1531
6,ESM+ALL_EXPLICIT,0.451876,1713
7,ESM+Atomic+Chem+MECH1,0.451778,1353
8,ESM+Atomic+Chem,0.447275,1319
9,ESM+Chem,0.443153,1304



Fold-level representation statistics:


,representation,mean,std,count,se,feature_dim
8,ESM+Atomic+Chem+MULTI2,0.458725,0.023387,5,0.010459,1377
6,ESM+Atomic+Chem+MECH1+MULTI2+SeqCtx,0.456484,0.029812,5,0.013332,1593
5,ESM+Atomic+Chem+MECH1+MULTI2,0.456407,0.031176,5,0.013942,1411
10,ESM+Atomic+Chem+MULTI2+Structure,0.455167,0.027536,5,0.012314,1497
9,ESM+Atomic+Chem+MULTI2+SeqCtx,0.454201,0.031020,5,0.013873,1559
7,ESM+Atomic+Chem+MECH1+MULTI2+Structure,0.452418,0.032149,5,0.014378,1531
1,ESM+ALL_EXPLICIT,0.451876,0.030212,5,0.013511,1713
4,ESM+Atomic+Chem+MECH1,0.451778,0.030730,5,0.013743,1353
3,ESM+Atomic+Chem,0.447275,0.028709,5,0.012839,1319
11,ESM+Chem,0.443153,0.030960,5,0.013846,1304



BEST_MEAN_REP = ESM+Atomic+Chem+MULTI2
ONE_SE_REP   = ESM+Atomic+Chem+MECH1
LOCK POLICY  = best_mean
LOCKED_REP   = ESM+Atomic+Chem+MULTI2
LOCKED_BLOCKS= ['ESM', 'ATOM', 'CHEM', 'METALSPEC', 'ELEC3D', 'TOPO', 'SOLV', 'SEQMOTIF']
LOCKED_DIM   = 1377


# Stage D — architecture/model-family screen

Only **after the representation is frozen** do we ask whether the learner itself should change.

Matched input, matched official folds:

- MLP
- KAN
- XGBoost

This cleanly separates:

```text
feature/representation gain
```

from:

```text
classifier/architecture gain
```

The winning model family is again selected from Dev CV only.


In [ ]:
# ============================================================
# 9. DEV-ONLY MODEL FAMILY SCREEN ON THE LOCKED REPRESENTATION
# ============================================================
MODEL_CANDIDATES = ['MLP','XGBoost']
if core.HAS_KAN:
    MODEL_CANDIDATES.insert(1, 'KAN')

LOCKED_GRID = {
    LOCKED_REP: LOCKED_BLOCKS
}

if RUN_MODEL_FAMILY_SCREEN:
    model_cv, model_folds = suite.run_dev_cv_only(
        ctx,
        LOCKED_GRID,
        models=tuple(MODEL_CANDIDATES),
        run_version=RUN_VERSION_MODEL,
    )

    display(model_cv.sort_values('CV_AUPRC', ascending=False))
    model_cv.to_csv(RUN_ROOT/'model_family_cv.csv', index=False)
    model_folds.to_csv(RUN_ROOT/'model_family_folds.csv', index=False)
else:
    model_cv = pd.read_csv(RUN_ROOT/'model_family_cv.csv')
    model_folds = pd.read_csv(RUN_ROOT/'model_family_folds.csv')

model_rank = model_cv.sort_values(
    'CV_AUPRC',
    ascending=False
).reset_index(drop=True)

LOCKED_MODEL = str(model_rank.iloc[0]['model'])

print('LOCKED MODEL FROM DEV =', LOCKED_MODEL)
display(model_rank)

model_decision = {
    **representation_decision,
    'candidate_models': MODEL_CANDIDATES,
    'locked_model': LOCKED_MODEL,
    'locked_model_cv_auprc': float(model_rank.iloc[0]['CV_AUPRC']),
}

with open(RUN_ROOT/'locked_model.json', 'w') as f:
    json.dump(model_decision, f, indent=2)


DONE MLP ESM+Atomic+Chem+MULTI2 1 AUPRC 0.45690187999406084
DONE MLP ESM+Atomic+Chem+MULTI2 2 AUPRC 0.4593231506989582
DONE MLP ESM+Atomic+Chem+MULTI2 3 AUPRC 0.4758339604455873
DONE MLP ESM+Atomic+Chem+MULTI2 4 AUPRC 0.4804939979113211
DONE MLP ESM+Atomic+Chem+MULTI2 5 AUPRC 0.4210740693277137
DONE KAN ESM+Atomic+Chem+MULTI2 1 AUPRC 0.44305388605347323
DONE KAN ESM+Atomic+Chem+MULTI2 2 AUPRC 0.45923972777164246
DONE KAN ESM+Atomic+Chem+MULTI2 3 AUPRC 0.47847561917291376
DONE KAN ESM+Atomic+Chem+MULTI2 4 AUPRC 0.48034449747918806
DONE KAN ESM+Atomic+Chem+MULTI2 5 AUPRC 0.4085972847388839


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [00:57:33] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


DONE XGBoost ESM+Atomic+Chem+MULTI2 1 AUPRC 0.4517006919351322
DONE XGBoost ESM+Atomic+Chem+MULTI2 2 AUPRC 0.4170860369909915
DONE XGBoost ESM+Atomic+Chem+MULTI2 3 AUPRC 0.4736337442616605
DONE XGBoost ESM+Atomic+Chem+MULTI2 4 AUPRC 0.4819100028497558
DONE XGBoost ESM+Atomic+Chem+MULTI2 5 AUPRC 0.4226210350837692


,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
1,MLP,ESM+Atomic+Chem+MULTI2,0.458725,0.023387,0.875499
0,KAN,ESM+Atomic+Chem+MULTI2,0.453942,0.029588,0.882158
2,XGBoost,ESM+Atomic+Chem+MULTI2,0.449390,0.029201,0.873081


LOCKED MODEL FROM DEV = MLP


,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
0,MLP,ESM+Atomic+Chem+MULTI2,0.458725,0.023387,0.875499
1,KAN,ESM+Atomic+Chem+MULTI2,0.453942,0.029588,0.882158
2,XGBoost,ESM+Atomic+Chem+MULTI2,0.449390,0.029201,0.873081


# Stage E — optional graph diagnostic

This is intentionally separated from the tabular lock.

The existing graph backend uses 18 relation descriptors including:

- Cα distance/radial kernels;
- sequence separation;
- canonical donor identity;
- donor–donor atom distance;
- first-shell coordination relation;
- donor↔polar second-shell support;
- acidic/basic/opposite-charge relations.

Two experiments are exposed:

1. **explicit-only residual graph** — tests whether relations add signal beyond handcrafted local descriptors;
2. **ESM locked residual graph** — optional, more expensive, tests whether graph relations add on top of the final ESM representation.

Graph results should be interpreted as a second-stage architecture experiment, not as feature screening.


In [ ]:
# ============================================================
# 10. OPTIONAL GRAPH DIAGNOSTIC
# ============================================================
graph_cv = None
graph_ext = None

if RUN_GRAPH_DIAGNOSTIC:
    explicit_locked = [
        b for b in LOCKED_BLOCKS
        if b != 'ESM'
    ]

    # AA keeps residue identity available when ESM is removed.
    explicit_node_blocks = ['AA'] + [
        b for b in explicit_locked
        if b != 'AA'
    ]

    GRAPH_EXPERIMENTS = {
        'ResidualGraph_ExplicitLocked_Rich': {
            'blocks': explicit_node_blocks,
            'architecture': 'residual',
            'edge_mode': 'rich',
        },
        'ResidualGraph_ExplicitLocked_Donor': {
            'blocks': explicit_node_blocks,
            'architecture': 'residual',
            'edge_mode': 'donor',
        },
    }

    if RUN_ESM_GRAPH:
        GRAPH_EXPERIMENTS['ResidualGraph_ESMLocked_Rich'] = {
            'blocks': LOCKED_BLOCKS,
            'architecture': 'residual',
            'edge_mode': 'rich',
        }
        GRAPH_EXPERIMENTS['ResidualGraph_ESMLocked_Donor'] = {
            'blocks': LOCKED_BLOCKS,
            'architecture': 'residual',
            'edge_mode': 'donor',
        }

    graph_cv, graph_ext = graph.run_graph_grid(
        ctx.stores,
        ctx.fold_ids,
        BASE,
        GRAPH_EXPERIMENTS,
        run_version=RUN_VERSION_GRAPH,
        cfg=graph.GraphConfig(),
        device=DEVICE,
    )

    display(graph_cv.sort_values('CV_AUPRC', ascending=False))
    display(graph_ext.sort_values(['dataset','AUPRC'], ascending=[True,False]))

    graph_cv.to_csv(RUN_ROOT/'graph_cv.csv', index=False)
    graph_ext.to_csv(RUN_ROOT/'graph_external_exploratory.csv', index=False)

else:
    print('Graph diagnostic OFF. This is recommended for the first feature-selection run.')


Graph diagnostic OFF. This is recommended for the first feature-selection run.


# Stage F — final locked external confirmation

At this point:

- representation is locked from Dev;
- model family is locked from Dev;
- thresholding is handled by the standardized backend using pooled Dev OOF predictions;
- no feature/model choice is allowed to depend on Test300/Test46.

The final run trains the same five official folds and reports the ensemble on both external sets.


In [ ]:
# ============================================================
# 11. FINAL LOCKED EXTERNAL RUN
# ============================================================
final_cv = None
final_ext = None

if RUN_FINAL_EXTERNAL:
    final_ctx = suite.clone_context_for_run(
        ctx,
        run_version=RUN_VERSION_FINAL,
        models=(LOCKED_MODEL,),
    )

    final_grid = {
        LOCKED_REP: LOCKED_BLOCKS
    }

    final_cv, final_ext, _ = suite.run_grid(
        final_ctx,
        final_grid,
        models=(LOCKED_MODEL,),
    )

    display(final_cv)
    display(final_ext.sort_values(
        ['dataset','AUPRC'],
        ascending=[True,False]
    ))

    final_cv.to_csv(RUN_ROOT/'FINAL_locked_cv.csv', index=False)
    final_ext.to_csv(RUN_ROOT/'FINAL_locked_external.csv', index=False)

else:
    print('Final external run OFF.')


RUN HASH: 5c13eb618253
RUN ROOT: /content/drive/MyDrive/Protein-binding-Phi/results/metal_v2_06_final_locked_external/metal_binding
RUN MLP | ESM+Atomic+Chem+MULTI2 | fold 1/5 | train=808 val=202
DONE MLP | ESM+Atomic+Chem+MULTI2 | fold 1: AUPRC=0.4569 AUROC=0.8406 F1@0.5=0.4500 (0.5 min)
RUN MLP | ESM+Atomic+Chem+MULTI2 | fold 2/5 | train=807 val=203
DONE MLP | ESM+Atomic+Chem+MULTI2 | fold 2: AUPRC=0.4593 AUROC=0.9024 F1@0.5=0.2495 (0.3 min)
RUN MLP | ESM+Atomic+Chem+MULTI2 | fold 3/5 | train=807 val=203
DONE MLP | ESM+Atomic+Chem+MULTI2 | fold 3: AUPRC=0.4758 AUROC=0.8852 F1@0.5=0.2880 (0.3 min)
RUN MLP | ESM+Atomic+Chem+MULTI2 | fold 4/5 | train=808 val=202
DONE MLP | ESM+Atomic+Chem+MULTI2 | fold 4: AUPRC=0.4805 AUROC=0.8727 F1@0.5=0.2425 (0.3 min)
RUN MLP | ESM+Atomic+Chem+MULTI2 | fold 5/5 | train=810 val=200
DONE MLP | ESM+Atomic+Chem+MULTI2 | fold 5: AUPRC=0.4211 AUROC=0.8766 F1@0.5=0.2882 (0.3 min)

=== CV SUMMARY (selection should be based here) ===
model         representat

,model,representation,folds,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,CV_F1_at_0_5,CV_MCC_at_0_5,mean_best_epoch,n_params,mean_seconds
0,MLP,ESM+Atomic+Chem+MULTI2,5,0.458725,0.023387,0.875499,0.303638,0.336113,4.2,369793.0,20.614607


,model,representation,dataset,n_residues,positive_residues,AUPRC,AUROC,F1_at_0.5,MCC_at_0.5,precision_at_0.5,...,at_0.5_macro_positive_mcc,at_0.5_macro_positive_mcc_ci95,at_0.5_n_macro_positive,at_0.5_n_false_alarm_only,at_OOF_macro_positive_f1,at_OOF_macro_positive_f1_ci95,at_OOF_macro_positive_mcc,at_OOF_macro_positive_mcc_ci95,at_OOF_n_macro_positive,at_OOF_n_false_alarm_only
0,MLP,ESM+Atomic+Chem+MULTI2,TestSet300,62689,881,0.381428,0.837568,0.313983,0.322743,0.231432,...,0.198958,0.035475,272,148,0.357612,0.064877,0.364292,0.065563,152,28
1,MLP,ESM+Atomic+Chem+MULTI2,TestSetNew46,6227,77,0.685333,0.969393,0.435088,0.480610,0.298077,...,0.188226,0.107263,39,24,0.510103,0.225424,0.515771,0.226553,17,2


# Stage G — compare against the four locked baselines already reproduced

This cell **does not run baseline models again**.

It only reads the frozen baseline result files from:

```text
baseline_reproduction_master_v6/results/
```

and shows the metal-only rows beside the new GEAR result when schemas allow it.

Interpretation:

- bindEmbed21DL + bindNode24 = direct benchmark competitors;
- ESMBind + MetalBind = metal-specific locked-checkpoint transfer baselines;
- GEAR = your DevSet1010-trained model.


In [ ]:
# ============================================================
# 12. READ LOCKED BASELINE RESULTS — NO BASELINE INFERENCE
# ============================================================
BASELINE_ROOT = BASE/'baseline_reproduction_master_v6'/'results'

baseline_files = {
    'direct': BASELINE_ROOT/'FULL_DIRECT_BASELINE_METRICS_300_46.csv',
    'ESMBind': BASELINE_ROOT/'ESMBind_TRANSFER_FULL_METRICS_300_46.csv',
    'MetalBind': BASELINE_ROOT/'MetalBind_TRANSFER_FULL_METRICS_300_46.csv',
}

baseline_frames = []

for group, fp in baseline_files.items():
    if not fp.exists():
        print('Missing:', fp)
        continue

    z = pd.read_csv(fp)

    if 'operating_point' in z.columns:
        z = z[z['operating_point'].eq('fixed_0.5')].copy()

    if 'label' in z.columns:
        z = z[z['label'].eq('metal_binding')].copy()

    z['baseline_group'] = group
    baseline_frames.append(z)

if baseline_frames:
    locked_baselines = pd.concat(
        baseline_frames,
        ignore_index=True,
        sort=False
    )
    locked_baselines.to_csv(
        RUN_ROOT/'locked_baseline_metal_rows.csv',
        index=False
    )

    show_cols = [
        c for c in [
            'method','dataset','label','AP_AUPRC','AUPRC','AUROC',
            'global_positive_F1','global_MCC',
            'protein_mean_F1','protein_mean_MCC'
        ]
        if c in locked_baselines.columns
    ]

    print('LOCKED BASELINES — METAL ONLY')
    display(locked_baselines[show_cols])

if final_ext is not None:
    print('\nOUR FINAL LOCKED EXTERNAL RESULT')
    display(final_ext.sort_values(['dataset','AUPRC'], ascending=[True,False]))


LOCKED BASELINES — METAL ONLY


,method,dataset,label,AP_AUPRC,AUROC,global_positive_F1,global_MCC,protein_mean_F1,protein_mean_MCC
0,bindEmbed21DL,TestSet300,metal_binding,0.307221,0.823064,0.360673,0.351863,0.219101,0.222255
1,bindEmbed21DL,TestSetNew46,metal_binding,0.613224,0.961886,0.554455,0.564205,0.261565,0.260439
2,bindNode24,TestSet300,metal_binding,0.321109,0.833192,0.402652,0.395457,0.257202,0.262066
3,bindNode24,TestSetNew46,metal_binding,0.621873,0.966996,0.605714,0.604670,0.286905,0.286599
4,ESMBind_7ion_max_TRANSFER,TestSet300,metal_binding,0.343212,0.832758,0.416911,0.409144,0.266866,0.270833
5,ESMBind_7ion_max_TRANSFER,TestSetNew46,metal_binding,0.615024,0.963630,0.577114,0.587362,0.318503,0.320249
6,MetalBind_13ion_max_TRANSFER,TestSet300,metal_binding,0.202519,0.826286,0.263556,0.282072,0.189624,0.192436
7,MetalBind_13ion_max_TRANSFER,TestSetNew46,metal_binding,0.342315,0.926658,0.253700,0.328136,0.184464,0.192525



OUR FINAL LOCKED EXTERNAL RESULT


,model,representation,dataset,n_residues,positive_residues,AUPRC,AUROC,F1_at_0.5,MCC_at_0.5,precision_at_0.5,...,at_0.5_macro_positive_mcc,at_0.5_macro_positive_mcc_ci95,at_0.5_n_macro_positive,at_0.5_n_false_alarm_only,at_OOF_macro_positive_f1,at_OOF_macro_positive_f1_ci95,at_OOF_macro_positive_mcc,at_OOF_macro_positive_mcc_ci95,at_OOF_n_macro_positive,at_OOF_n_false_alarm_only
0,MLP,ESM+Atomic+Chem+MULTI2,TestSet300,62689,881,0.381428,0.837568,0.313983,0.322743,0.231432,...,0.198958,0.035475,272,148,0.357612,0.064877,0.364292,0.065563,152,28
1,MLP,ESM+Atomic+Chem+MULTI2,TestSetNew46,6227,77,0.685333,0.969393,0.435088,0.480610,0.298077,...,0.188226,0.107263,39,24,0.510103,0.225424,0.515771,0.226553,17,2


# Stage H — final scientific digest

The notebook ends by writing a compact decision record:

- dataset counts;
- feature-family ranking;
- component ranking;
- combination ranking;
- selected representation;
- selected model family;
- final external results;
- environment snapshot.

This is the file to keep with the manuscript experiment archive.


In [ ]:
# ============================================================
# 13. FINAL DIGEST / MANIFEST
# ============================================================
digest = {
    'gear_ultimate_version': GEAR_ULTIMATE_VERSION,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'device': str(DEVICE),
    'primary_metric': 'AUPRC',
    'development_set': 'DevSet1010 official 5-fold protein split',
    'external_sets': ['TestSet300','TestSetNew46'],
    'selection_policy': SELECTION_POLICY,
    'locked_representation': LOCKED_REP,
    'locked_blocks': LOCKED_BLOCKS,
    'locked_dimension': block_dim(LOCKED_BLOCKS),
    'locked_model': LOCKED_MODEL,
    'graph_diagnostic_run': bool(RUN_GRAPH_DIAGNOSTIC),
    'final_external_run': bool(RUN_FINAL_EXTERNAL),
}

with open(RUN_ROOT/'FINAL_DECISION_MANIFEST.json', 'w') as f:
    json.dump(digest, f, indent=2)

print(json.dumps(digest, indent=2))

print('\nArtifacts:')
for fp in sorted(RUN_ROOT.glob('*')):
    print(' -', fp.name)

print('\n' + '='*90)
print('GEAR METAL-ONLY ULTIMATE V2: PIPELINE COMPLETE')
print('Feature selection source: DevSet1010 only')
print('Locked representation:', LOCKED_REP)
print('Locked model:', LOCKED_MODEL)
if RUN_FINAL_EXTERNAL:
    print('External confirmation: TestSet300 + TestSetNew46 complete')
print('='*90)


{
  "gear_ultimate_version": "1.0.0",
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "device": "cuda",
  "primary_metric": "AUPRC",
  "development_set": "DevSet1010 official 5-fold protein split",
  "external_sets": [
    "TestSet300",
    "TestSetNew46"
  ],
  "selection_policy": "best_mean",
  "locked_representation": "ESM+Atomic+Chem+MULTI2",
  "locked_blocks": [
    "ESM",
    "ATOM",
    "CHEM",
    "METALSPEC",
    "ELEC3D",
    "TOPO",
    "SOLV",
    "SEQMOTIF"
  ],
  "locked_dimension": 1377,
  "locked_model": "MLP",
  "graph_diagnostic_run": false,
  "final_external_run": true
}

Artifacts:
 - FINAL_DECISION_MANIFEST.json
 - FINAL_locked_cv.csv
 - FINAL_locked_external.csv
 - combo_ladder_cv.csv
 - combo_ladder_folds.csv
 - component_screen_cv.csv
 - component_screen_folds.csv
 - component_screen_ranked.csv
 - family_screen_cv.csv
 - family_screen_folds.csv
 - family_screen_ranked.csv
 - locked_baseline_metal_rows.csv
 - locked_model.json
 - locked_representation.json
 -